In [51]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
df_train =  pd.read_parquet("all_teams_last10seasons_with_opponent_rolls.parquet")
df_test = pd.read_parquet("demo.parquet")

In [52]:
target = 'TARGET_WL'

bool_cols = df_train.select_dtypes(include=['bool']).columns
if len(bool_cols) > 0:
    df_train[bool_cols] = df_train[bool_cols].astype(int)
    df_test[bool_cols] = df_test[bool_cols].astype(int)
    
    
# Features = all numeric columns except the target
feature_cols = df_train.drop(columns=[target]).select_dtypes(include=[np.number]).columns

X_train = df_train[feature_cols]
y_train = df_train[target]

X_test = df_test[feature_cols]
y_test = df_test[target]

In [53]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# --- 1. Final tuned Logistic Regression pipeline ---
final_log_model = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(
        max_iter=2000,
        penalty='l1',       # tuned best param
        C=0.01,             # tuned best param
        solver='saga',      # tuned best param
        l1_ratio=0          # tuned best param (only for elasticnet, saga allows l1)
    ))
])

# Fit on training data
final_log_model.fit(X_train, y_train)

# Predict on test data
y_pred_log = final_log_model.predict(X_test)


/Users/joshuademontigny/Downloads/ML_NBA_Project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(


In [54]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# --- SVM (tuned) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm_model = SVC(
    C=1,
    gamma=0.001,
    kernel='rbf',
    probability=True,
    random_state=42
)

svm_model.fit(X_train_scaled, y_train)
y_pred_svm = svm_model.predict(X_test_scaled)

In [55]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# --- Random Forest (tuned) ---
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight=None,
    random_state=42
)

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

In [56]:
import os
from xgboost import XGBClassifier

# --- 1️⃣ Train XGBoost using tuned hyperparameters ---
xgb_model = XGBClassifier(
    objective="binary:logistic",  # binary classification
    eval_metric="logloss",
    tree_method="hist",           # fast on Apple Silicon
    random_state=42,
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=1.0,
    colsample_bytree=0.8
)

xgb_model.fit(X_train, y_train)

# --- 2️⃣ Predict ---
y_pred_xgb = xgb_model.predict(X_test)        # predicted class labels
y_proba_xgb = xgb_model.predict_proba(X_test) # predicted probabilities for soft voting

In [57]:
output_folder = "tables"
os.makedirs(output_folder, exist_ok=True)

results = []

# Logistic Regression
results.append({
    "model": "Logistic Regression (tuned)",
    "accuracy": round(accuracy_score(y_test, y_pred_log), 2),
    "precision": round(precision_score(y_test, y_pred_log, average='macro'), 2),
    "recall": round(recall_score(y_test, y_pred_log, average='macro'), 2),
    "f1_score": round(f1_score(y_test, y_pred_log, average='macro'), 2)
})

# SVM
results.append({
    "model": "SVM (tuned)",
    "accuracy": round(accuracy_score(y_test, y_pred_svm), 2),
    "precision": round(precision_score(y_test, y_pred_svm, average='macro'), 2),
    "recall": round(recall_score(y_test, y_pred_svm, average='macro'), 2),
    "f1_score": round(f1_score(y_test, y_pred_svm, average='macro'), 2)
})

# Random Forest
results.append({
    "model": "Random Forest (tuned)",
    "accuracy": round(accuracy_score(y_test, y_pred_rf), 2),
    "precision": round(precision_score(y_test, y_pred_rf, average='macro'), 2),
    "recall": round(recall_score(y_test, y_pred_rf, average='macro'), 2),
    "f1_score": round(f1_score(y_test, y_pred_rf, average='macro'), 2)
})

# XGBoost
results.append({
    "model": "XGBoost (tuned)",
    "accuracy": round(accuracy_score(y_test, y_pred_xgb), 2),
    "precision": round(precision_score(y_test, y_pred_xgb, average='macro'), 2),
    "recall": round(recall_score(y_test, y_pred_xgb, average='macro'), 2),
    "f1_score": round(f1_score(y_test, y_pred_xgb, average='macro'), 2)
})


# --- SOFT VOTING ENSEMBLE ---
y_proba_log = final_log_model.predict_proba(X_test)
y_proba_svm = svm_model.predict_proba(X_test_scaled)
y_proba_rf = rf_model.predict_proba(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)

# ---- Simple average (soft voting with XGB) ----
avg_proba = (y_proba_log + y_proba_svm + y_proba_rf + y_proba_xgb) / 4
y_pred_ensemble_soft = np.argmax(avg_proba, axis=1)


# Compute metrics
soft_result = {
    "model": "Ensemble Soft Voting",
    "accuracy": round(accuracy_score(y_test, y_pred_ensemble_soft), 2),
    "precision": round(precision_score(y_test, y_pred_ensemble_soft, average='macro'), 2),
    "recall": round(recall_score(y_test, y_pred_ensemble_soft, average='macro'), 2),
    "f1_score": round(f1_score(y_test, y_pred_ensemble_soft, average='macro'), 2)
}

# ---- 2️⃣ Weighted soft voting ----
weights = [0.4, 0.15, 0.15, 0.3]
weighted_avg_proba = (
    weights[0]*y_proba_log +
    weights[1]*y_proba_svm +
    weights[2]*y_proba_rf +
    weights[3]*y_proba_xgb
) / sum(weights)
y_pred_ensemble_weighted = np.argmax(weighted_avg_proba, axis=1)


weighted_result = {
    "model": "Ensemble Weighted Soft Voting",
    "accuracy": round(accuracy_score(y_test, y_pred_ensemble_weighted), 2),
    "precision": round(precision_score(y_test, y_pred_ensemble_weighted, average='macro'), 2),
    "recall": round(recall_score(y_test, y_pred_ensemble_weighted, average='macro'), 2),
    "f1_score": round(f1_score(y_test, y_pred_ensemble_weighted, average='macro'), 2)
}

# --- Remove any previous ensemble entries if rerunning ---
results = [r for r in results if "Ensemble" not in r["model"]]

# --- Append new ensemble results ---
results.extend([soft_result, weighted_result])

# --- Convert to DataFrame and save ---
comparison_table = pd.DataFrame(results)
comparison_table.to_csv(os.path.join(output_folder, "model_comparison.csv"), index=False)

print("✅ Updated model comparison table saved to tables/model_comparison.csv")
print(comparison_table)

✅ Updated model comparison table saved to tables/model_comparison.csv
                           model  accuracy  precision  recall  f1_score
0    Logistic Regression (tuned)      0.68       0.68    0.68      0.68
1                    SVM (tuned)      0.68       0.68    0.68      0.68
2          Random Forest (tuned)      0.68       0.68    0.68      0.68
3                XGBoost (tuned)      0.69       0.69    0.69      0.69
4           Ensemble Soft Voting      0.69       0.69    0.69      0.69
5  Ensemble Weighted Soft Voting      0.68       0.68    0.68      0.68
